#BERTopic on UDC (URDU Document Clustering) Dataset
In this notebook we have performed hyperparameter tunning on Urdu News Data using a bert based topic modelling technique BERTopic.

## Mounting Google Drive
If the dataset is on Google Drive then you have to mount over google drive with collaboratory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive




#Installing required dependencies
**One thing to remember is that after installing libraries you have to restart the run time again so that other dependencies are not affected by it.**

In [ ]:
!pip install bertopic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 66.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.9/90.9 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 10.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached Cython-0.29.36-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux_2_24_x86_64.whl (1.9 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 6.6 MB/s eta 0:00:00
  Created wheel for hdbscan: filename=hdbscan-0.8.33-cp310-cp310-linux_x86_64.whl size=3039179 sha256=4f6a93bc5d483c9c47a45f1da64bf8a6b66c41ba44744de57a33098a2227b83a
  Stored in di

In [ ]:
!pip install -U sentence-transformers

In [ ]:
!pip install --upgrade tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 32.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 33.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.9 MB/s eta 0:00:00
  Attempting uninstall: tensorflow-estimator
    Found existing installation: tensorflow-estimator 2.12.0
    Uninstalling tensorflow-estimator-2.12.0:
      Successfully uninstalled tensorflow-estimator-2.12.0
  Attempting uninstall: keras
    Found existing installation: keras 2.12.0
    Uninstalling keras-2.12.0:
      Successfully uninstalled keras-2.12.0
  Attempting uninstall: google-auth-oauthlib
    Found existing installation: google-auth-oauthlib 0.4.6
    Uninstalling google-auth-oauthlib-0.4.6:
      Successfully uninstalled google-auth-oauthlib-0.4.6
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.12.0
    Uninstalling te


# Importing required dependencies
We will import numpy, pandas and re, bertopic, gensim library for now. other libraries will be imported in the notebook later.

Pandas will be used to create a Dataframe and handle the csv file. Numpy will be used for the faster computation of arrays to save time. re library will be used for the cleaning of data. gensim library is used to get coherence score. bertopic is used to train bertopic on our Bigcorpus dataset with using pretrained language models

In [ ]:
import pandas as pd
import numpy as np
import re
from bertopic import BERTopic
from gensim.models.coherencemodel import CoherenceModel
import gensim.corpora as corpora
#optional
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.ERROR)

import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning)

##DataFrame
We also performed experiment on publically available dataset for topic modelling that consist of 5 categories=>Business, Entertainment, Health,weird and Sports. After cleaning this news, total 1008 rows saved into another new column Clean_Data for experiments.The link to the dataset is given below:

[Dataset](https://github.com/Mubashar331/Urdu-corpus)




In [ ]:
#load UNTM Dataset
df = pd.read_csv('/content/drive/MyDrive/Clean_BigCorpus.csv', encoding='utf-8')

In [ ]:
print(len(df))
df.head()

1008


,Document,Category,Clean_Data
0,﻿اسلام آباد: ایسوسی ایشن آف چارٹرڈ سرٹیفائی...,Business,اسلام آباد ایسوسی ایشن آف چارٹرڈ سرٹیفائیڈا...
1,اسلام آباد: ایف بی آرنے 1 ارب30کروڑ روپے سے ...,Business,اسلام آباد ایف بی آرنے اربکروڑ روپے سے زائد...
2,کراچی: بزنس مین گروپ کے چیئرمین اور کراچی چیم...,Business,کراچی بزنس مین گروپ کے چیئرمین اور کراچی چیمب...
3,کراچی: ٹرانسپورٹ اور پٹرولیم مصنوعات کی درآم...,Business,کراچی ٹرانسپورٹ اور پٹرولیم مصنوعات کی درآمد...
4,بجٹ کی آمد آمد ہے۔ ایکسپریس میڈیا گروپ کی ی...,Business,بجٹ کی آمد آمد ہے۔ ایکسپریس میڈیا گروپ کی ی...


## Preprocessing of Data
Data was cleaned that saved into column "Clean_Data" we  removed some extra coloumns first. Because we do experiment only news  and all other things are extra for us.

Stopwords are common words that are often filtered out during text processing in natural language processing (NLP) tasks. These words are considered to have little or no value in conveying the actual meaning of the text. We take list of 401 stopwords for topic modelling.

In [ ]:
df['Clean_Data'] = df['Clean_Data'].str.replace('شیئرٹویٹشیئرای میلتبصرے مزید شیئر', '')

In [ ]:
df['Clean_Data'] = df['Clean_Data'].str.replace('کیلیے', '')

In [ ]:
def remove_nonbreaking_space(text):
    return re.sub(r'\xa0', ' ', text)

df['Clean_Data'] = df['Clean_Data'].apply(remove_nonbreaking_space)

In [ ]:
documents = df['Clean_Data'].values.tolist()
print((documents[1]))

 اسلام آباد ایف بی آرنے  اربکروڑ روپے سے زائد مالیت کے گڈز ڈکلیئریشن فنڈ کے استعمال  خصوصی سیکریٹریٹ قائم کرنے اور ایف بی آر ہیڈ کوارٹرز اورکسٹمز ہاؤس کراچی میں قائم ڈیٹا سینٹرزکی اپ گریڈیشن کرنے کا فیصلہ کیا ہے۔  ایکسپریس کو دستیاب دستاویزکے مطابق یہ اہم فیصلے ایف بی آرکے ممبر کسٹمز زاہدکھوکھرکی زیرصدارت اجلاس میں کیے گئے جس میں یہ بھی فیصلہ کیا گیا کہ کسٹمزکراچی کے وی بوک اوراس کے نیٹ ورک ڈیٹا سینٹر کی فعالیت کو برقرار رکھنے کے لیے پاکستان ریونیو آٹومیشن لمیٹڈ کی جانب سے ضرورت کے مطابق بروقت پروکیورمنٹس کو بھی یقینی بنایا جائے گا تاکہ آئندہ کسٹمز کراچی کے وی بوک نیٹ ورک ڈیٹا سینٹر میں بریک ڈاون کی صورتحال سے بچا جا سکے۔  اجلاس میں یہ بھی فیصلہ کیا گیا کہ اربکروڑ روپے سے زائد مالیت کے گڈز ڈکلیئریشن فنڈ کے استعمال کے لیے خصوصی سیکریٹریٹ قائم کیا جائے گا اور یہ سیکریٹریٹ دسمبر کو جاری کردہ کسٹمز جنرل آرڈر نمبر کے مطابق جی ڈی فنڈ کو استعمال میں لائے گا۔ اس بارے میں ایف بی آر حکام کا کہنا ہے کہ ایف بی آر نے  میں درآمدی کنسائنمنٹس کی پروسیسنگ پر  روپے فی کنسائنمنٹ کے حساب سے گڈز ڈ

In [ ]:
import nltk
stopwords=pd.read_csv('/content/drive/MyDrive/stopwords.txt',names=['List'])
# stopwords['List']

stopwords_ur=[]
for li in stopwords['List']:
  stopwords_ur.append(li)
print(len(stopwords_ur))
print(stopwords_ur)

401
['کی', 'ہیں', 'ہے', 'رہا', 'رہی', 'رہے', 'تھا', 'تھے', 'تھی', 'تھى', 'میں', 'کہ', 'کے', 'ہوتے', 'کہہ', 'بنانا', 'پھر', 'لکین', 'ہوتی', 'لیتی', 'ایسی', 'ائے', 'ہوئے', 'ہوۓ ', 'مگر', 'چاہے', 'کیے', 'تاکہ', 'تم', 'آکر', 'لگا', 'ہوگیئں', 'ليے', 'رہتی', 'ہوگئی', 'انھوں', 'چاہتے', 'پاگیئں', 'آنا', 'کروا', 'رکھ', 'آتی', 'یہاں', 'جیسا', 'چکے', 'کرئے', 'دیے', 'چکا', 'ملتا', 'کبھی', 'ایسا', 'کرسکیں', 'ہوسکے', 'سکیں', 'لہذا', 'چاہئے', 'وہیں', 'اٹھایا', 'جہنوں', 'ہمارے', 'لیے', 'آرہے', 'کرگیئں', 'بنانے', 'سکتا', 'وغیرہ', 'دے', 'ہوۓ', 'رہنے', 'ہوۓ', 'کئے', 'لگے', 'لگایا', 'لائے', 'کہے', 'کرے', 'رہئں', 'آگئے', 'کئی', 'کم', 'ملی', 'جنہیں', 'ہوئیں', 'تھیں', 'ابھی', 'پاگئیں', 'آئے', 'کرا', 'دیا', 'جہاں', 'آگئیں', 'کرتی', 'رہیں', 'کرتیں', 'دیتی', 'ہوگئے', 'ديتے', 'انہں', 'ایسے', 'رکھتے', 'رہتے', 'رکھی', 'کیں', 'كیا', 'وه', 'جنہيں', 'کروائی', 'دینا', 'جسے', 'جاتی', 'اسکی', 'ملے', 'کرگئى', 'جن', 'آپکو', 'جنہوں', 'دیئے', 'والی', 'جبکہ', 'دیگر', 'آپکا', 'رکھنے', 'انہىں', 'کيے', 'یعنی', 'كيے', 'بننے', 'ج

# BERTopic Training
The default  bertopic embedding model is paraphrase-multilingual-MiniLM-L12-v2 when selecting language="multilingual". we also performed experiment on 2 another pretained multilingual model from [sentence-tranformer](https://www.sbert.net/docs/pretrained_models.html) and create custom document embedding and passed it to the bertopic model for training.

In [ ]:
#create custom embedding
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')
embeddings = model.encode(documents, show_progress_bar=True)
print(embeddings)

.gitattributes:   0%|          | 0.00/690 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2_Dense/config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.69k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

[[ 0.05808477  0.0238898   0.00667638 ...  0.00750619  0.05943276
   0.00124432]
 [ 0.00707471  0.06183453  0.01358282 ...  0.03410887  0.04613194
  -0.01144732]
 [-0.04789835  0.05102404 -0.02981449 ...  0.0342866  -0.02764623
   0.00305767]
 ...
 [-0.01387034  0.02076214 -0.04047821 ...  0.00534513 -0.02091605
   0.01629284]
 [-0.05897831  0.0367678   0.03810614 ...  0.06076489 -0.00818423
   0.0283733 ]
 [ 0.0256409  -0.00437732 -0.04543339 ...  0.04750746 -0.03870325
   0.04952908]]


In [ ]:
#pass vectorizer_model to bertopic for stopwords removal
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(stop_words= stopwords_ur)


In [ ]:
#ClassTfidf for top words
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

In [ ]:
from umap import UMAP

In [ ]:
from hdbscan import HDBSCAN

In [ ]:
np.random.seed(42)

**Diversity Score**

In [ ]:
def topic_diversity(topics, topk=10):
    """
    compute the proportion of unique words

    Parameters
    ----------
    topics: a list of lists of words
    topk: top k words on which the topic diversity will be computed
    """
    if topk > len(topics[0]):
        raise Exception('Words in topics are less than '+str(topk))
    else:
        unique_words = set()
        for topic in topics:
            unique_words = unique_words.union(set(topic[:topk]))
        puw = len(unique_words) / (topk * len(topics))
        return puw

In [ ]:
import itertools
from rbo import rbo
import numpy as np

class InvertedRBO:
    def __init__(self):
        pass

    def irbo(self, topics, topk=10, weight=0.9):
        """
        Calculate inverted Rank Biased Overlap (RBO) as a measure of topic diversity from a list of lists of words.

        :param topics: A list of lists of words representing different topics.
        :param topk: The number of top words on which RBO will be computed.
        :param weight: Weight of each agreement at depth d: p**(d-1). When set to 1.0, there is no weight,
                       and the RBO returns to average overlap.
        :return: The inverted RBO topic diversity score.
        """
        if topk <= 0:
            raise ValueError("topk must be a positive integer.")

        num_topics = len(topics)
        if num_topics == 0:
            raise ValueError("topics list cannot be empty.")

        if topk > len(topics[0]):
            raise Exception('Words in topics are less than topk')

        collect = []
        for list1, list2 in itertools.combinations(topics, 2):
            rbo_val = rbo(list1[:topk], list2[:topk], p=weight)[2]
            collect.append(rbo_val)

        Irbo_score = 1 - np.mean(collect)
        return Irbo_score

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from itertools import product
from sklearn.cluster import AgglomerativeClustering

# cluster_model = AgglomerativeClustering(n_clusters=5)
# cluster_model = KMeans(n_clusters=5, random_state=42)
texts = [[word for word in str(document).split() if word not in stopwords_ur] for document in documents]
id2word = corpora.Dictionary(texts)
corpus = [id2word.doc2bow(text) for text in texts]

# Define lists for n_neighbors and n_components
n_neighbors_list = [5, 10, 15, 20,25, 30]
n_components_list = [2, 3, 4, 5]

# Create a list to store the data for each combination
data_list = []

# Loop over the parameter combinations and fit BERTopic models
for n_neighbors, n_components in product(n_neighbors_list, n_components_list):
    umap_param = {'n_neighbors': n_neighbors, 'n_components': n_components}

    # Create UMAP and HDBSCAN models with the current parameter combination
    umap_model = UMAP(**umap_param, random_state=42)

    # Fit a BERTopic model with the current parameter combination
    topic_model = BERTopic(language="urdu",
                          low_memory=True,
                          calculate_probabilities=True,
                          vectorizer_model=vectorizer_model,
                          top_n_words=10,
                          verbose=True, umap_model=umap_model, ctfidf_model=ctfidf_model, nr_topics=6)

    topics, probs = topic_model.fit_transform(documents, embeddings)

    topics_bert = []
    for i in topic_model.get_topics():
        row = []
        topic = topic_model.get_topic(i)
        for word in topic:
            row.append(word[0])
        topics_bert.append(row)

    # Calculate coherence scores using Gensim's Coherence Model
    coherence_cv = CoherenceModel(topics=topics_bert, texts=texts, dictionary=id2word, coherence='c_v')
    cv_score = round(coherence_cv.get_coherence(), 3)
    coherence_npmi = CoherenceModel(topics=topics_bert, texts=texts, dictionary=id2word, coherence='c_npmi')
    npmi_score = round(coherence_npmi.get_coherence(),3)

    # Calculate Unique Word count
    unique_words = round(topic_diversity(topics_bert, topk=10),3)

    # Calculate IRBO Score
    inverted_rbo_calculator = InvertedRBO()
    IRBO = round(inverted_rbo_calculator.irbo(topics_bert, topk=10, weight=0.9),3)

    # Append the data for the current combination to the list
    data_list.append({
        'Clustering': 'HDBSCAN',
        'Pretrained_Model': 'DistilUSE',
        'n_neighbors': n_neighbors,
        'n_components': n_components,
        'C_v score': cv_score,
        'C_NPMI score': npmi_score,
        'Unique Word': unique_words,
        'IRBO Score': IRBO,
        'Topics': topics_bert
    })

# Create a DataFrame from the data list
df1 = pd.DataFrame(data_list)

# Print the DataFrame
print(df1)


2023-11-29 13:50:12,843 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2023-11-29 13:50:17,156 - BERTopic - Dimensionality - Completed ✓
2023-11-29 13:50:17,160 - BERTopic - Cluster - Start clustering the reduced embeddings
2023-11-29 13:50:17,262 - BERTopic - Cluster - Completed ✓
2023-11-29 13:50:17,268 - BERTopic - Representation - Extracting topics from clusters using representation models.
2023-11-29 13:50:18,404 - BERTopic - Representation - Completed ✓
2023-11-29 13:50:18,410 - BERTopic - Topic reduction - Reducing number of topics
2023-11-29 13:50:18,417 - BERTopic - Topic reduction - Reduced number of topics from 3 to 3
2023-11-29 13:50:21,809 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2023-11-29 13:50:24,725 - BERTopic - Dimensionality - Completed ✓
2023-11-29 13:50:24,727 - BERTopic - Cluster - Start clustering the reduced embeddings
2023-11-29 13:50:24,791 - BERTopic - Cluster - Completed ✓
2023-11-29 13:50:24,

   Clustering Pretrained_Model  n_neighbors  n_components  C_v score  \
0     HDBSCAN        DistilUSE            5             2      0.734   
1     HDBSCAN        DistilUSE            5             3      0.680   
2     HDBSCAN        DistilUSE            5             4      0.734   
3     HDBSCAN        DistilUSE            5             5      0.626   
4     HDBSCAN        DistilUSE           10             2      0.734   
5     HDBSCAN        DistilUSE           10             3      0.733   
6     HDBSCAN        DistilUSE           10             4      0.734   
7     HDBSCAN        DistilUSE           10             5      0.734   
8     HDBSCAN        DistilUSE           15             2      0.670   
9     HDBSCAN        DistilUSE           15             3      0.621   
10    HDBSCAN        DistilUSE           15             4      0.669   
11    HDBSCAN        DistilUSE           15             5      0.734   
12    HDBSCAN        DistilUSE           20             2      0

In [ ]:
# Save the DataFrame to a CSV file
df1.to_csv('UDC_HDBSCAN_ClusteringF.csv', mode='a', index=False,  encoding='utf-8-sig')